In [10]:
import re
import gzip

# Tên file chứa mảng gzip (camera_index.h)
INPUT_FILE = "camera_index.h"
OUTPUT_FILE = "index.html"

# Đọc nội dung file .h
with open(INPUT_FILE, "r", encoding="utf-8", errors="ignore") as f:
    content = f.read()

# Tìm mảng dữ liệu hex (các số dạng 0x...)
match = re.search(r'\{(.+?)\}', content, re.DOTALL)
if not match:
    print("❌ Không tìm thấy dữ liệu mảng trong file .h!")
    exit()

# Lấy dữ liệu hex và chuyển thành byte array
hex_list = match.group(1).split(',')
byte_data = bytes([int(h.strip(), 16) for h in hex_list if h.strip().startswith('0x')])

# Giải nén gzip
try:
    html_data = gzip.decompress(byte_data)
except Exception as e:
    print(f"❌ Giải nén GZIP thất bại: {e}")
    exit()

# Ghi ra file index.html
with open(OUTPUT_FILE, "wb") as f:
    f.write(html_data)

print(f"✅ Đã giải nén thành công → {OUTPUT_FILE}")


✅ Đã giải nén thành công → index.html


In [1]:
import gzip
import shutil

# Đường dẫn file đầu vào và đầu ra
input_file = "index.html"
output_file = "index.html.gz"

# Mở file HTML và nén thành GZIP
with open(input_file, 'rb') as f_in:
    with gzip.open(output_file, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print(f"✅ Đã nén thành công: {output_file}")


✅ Đã nén thành công: index.html.gz


In [2]:
def to_c_array(data_bytes, array_name="index_ov2640_html_gz"):
    hex_lines = []
    for i in range(0, len(data_bytes), 20):
        chunk = data_bytes[i:i+20]
        hex_line = ', '.join(f'0x{b:02X}' for b in chunk)
        hex_lines.append(f'  {hex_line}')
    hex_data = ',\n'.join(hex_lines)
    return f"""\
#define {array_name}_len {len(data_bytes)}
const uint8_t {array_name}[] = {{
{hex_data}
}};
"""

# Tên file input/output
input_gz = "index.html.gz"
output_h = "camera_index1.h"

# Đọc dữ liệu gzip
with open(input_gz, "rb") as f:
    data = f.read()

# Tạo nội dung .h
c_header = to_c_array(data, array_name="index_ov2640_html_gz")

# Ghi ra file .h
with open(output_h, "w") as f:
    f.write(c_header)

print(f"✅ Đã tạo file {output_h} thành công từ {input_gz}")


✅ Đã tạo file camera_index1.h thành công từ index.html.gz
